In [3]:
from datasets import load_dataset
dataset=load_dataset("csv",data_files="ERUPD_NMT.csv")
dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['English', 'Roman Urdu'],
        num_rows: 10681
    })
})

In [4]:
dataset["train"][0]

{'English': 'In the heart of the bustling city, lived a young woman named Aisha',
 'Roman Urdu': 'Shehar ki dil mein rehti thi ek nojawan aurat jiska naam Aisha tha'}

# Data Cleaning

In [5]:
df=dataset["train"].to_pandas()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10681 entries, 0 to 10680
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   English     10681 non-null  object
 1   Roman Urdu  10680 non-null  object
dtypes: object(2)
memory usage: 167.0+ KB


In [6]:
# Looking for
df.duplicated().sum()
# approximately 10% data is duplicate that must be prohibited

np.int64(17)

In [7]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [8]:
print(df.isnull().sum())
df.dropna(inplace=True)
print(df.isnull().sum())

English       0
Roman Urdu    1
dtype: int64
English       0
Roman Urdu    0
dtype: int64


In [9]:
df

,English,Roman Urdu
0,"In the heart of the bustling city, lived a you...",Shehar ki dil mein rehti thi ek nojawan aurat ...
1,She had always dreamed of exploring the world ...,Usne hamesha khwab dekha tha ke apne sheher ke...
2,"One day, as she perused a travel magazine, Ais...","Ek din, jab usne ek safar nama parha, Aisha ne..."
3,The vivid descriptions of lush landscapes and ...,Yehan ke hari bhari manazir aur gaon walon ki ...
4,"Determined to turn her dreams into reality, Ai...",Apne khwabon ko haqeeqat mein tabdeel karne ka...
...,...,...
10675,"A Japanese man in a red shirt, at the Olympics...","Aik Japanese shakhs jo laal shirt mein hai, Ol..."
10676,A man throws the bowling ball to knock down th...,Aik shakhs bowling ball phenk raha hai taake a...
10677,A woman dressed in a traditional wrap can be s...,Aik aurat jo riwayati kapde pehni hui hai aik ...
10678,A woman carrying a bowl on her head is walking...,Aik aurat jo apne sar par aik katora le kar ba...


In [10]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(df_train.shape)
print(df_test.shape)

(8530, 2)
(2133, 2)


In [11]:
from datasets import Dataset, DatasetDict

# 1. Convert each Pandas DataFrame into a separate Hugging Face Dataset
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

# 2. Wrap them together into a single DatasetDict object
raw_datasets = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

# 3. Clean up the automatic Pandas index column if it exists
for split in raw_datasets.keys():
    if "__index_level_0__" in raw_datasets[split].column_names:
        raw_datasets[split] = raw_datasets[split].remove_columns(["__index_level_0__"])

# 4. Verify your final structure
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['English', 'Roman Urdu'],
        num_rows: 8530
    })
    test: Dataset({
        features: ['English', 'Roman Urdu'],
        num_rows: 2133
    })
})


In [12]:
# separating the romand urdu and english sentences for Seq2seq tokenizer
roman_urdu_sentences=raw_datasets["train"]["Roman Urdu"]
english_sentences=raw_datasets["train"]["English"]

print(roman_urdu_sentences[0])
print(english_sentences[0])

Safed qameez aur tan rang ke pant pehna hua shakhs chhat par naye shingles laga raha hain.
Man in white shirt and tan pants is putting new shingles on roof.


In [13]:
from transformers import AutoTokenizer
t5_tokenizer=AutoTokenizer.from_pretrained("google-t5/t5-base")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [14]:
from transformers import DataCollatorForSeq2Seq

# 1. Define the explicit -100 padding tokenization function
def tokenize_function(examples):
    inputs = ["translate Roman Urdu to English: " + str(text) for text in examples["Roman Urdu"]]
    targets = [str(text) for text in examples["English"]]

    # Tokenize with max_length padding enabled
    model_inputs = t5_tokenizer(
        inputs,
        text_target=targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Manually swap out target pad tokens (0) for -100
    labels = model_inputs["labels"]
    model_inputs["labels"] = [
        [(token if token != t5_tokenizer.pad_token_id else -100) for token in label]
        for label in labels
    ]

    return model_inputs

# Maping the updated function across all dataset splits
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/2133 [00:00<?, ? examples/s]

In [15]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

# Loading the pre-trained T5 model
model_name = "google-t5/t5-base"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [16]:
# 3. Clean up the raw text columns so the collator doesn't crash
cleaned_datasets = tokenized_datasets.remove_columns(["English", "Roman Urdu"])

# 4. Verify using your Data Collator check
data_collator = DataCollatorForSeq2Seq(tokenizer=t5_tokenizer, model=model)
batch = data_collator([cleaned_datasets["train"][i] for i in range(1, 3)])

#print("Collator Keys:", batch.keys())
print("Sample Labels Matrix (Checking for -100):\n", batch["labels"])

Sample Labels Matrix (Checking for -100):
 tensor([[   71,  1021,  2335,    19,  5119,     3,     9, 23348,  6033,    40,
          1067,     3,     9,  2078,     5,     1,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         

In [17]:
pip install evaluate sacrebleu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 16.1 MB/s eta 0:00:00


In [18]:
import evaluate
import numpy as np

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # 1. Handle cases where the model returns extra prediction logits
    if isinstance(preds, tuple):
        preds = preds[0]

    # 2. Replace -100 values in predictions with the pad token ID
    preds = np.where(preds != -100, preds, t5_tokenizer.pad_token_id)

    # 3. Replace -100 values in target labels
    labels = np.where(labels != -100, labels, t5_tokenizer.pad_token_id)

    # 4. FIX: Clip indices to be within the safe vocabulary range and cast explicitly to native python integers
    vocab_size = t5_tokenizer.vocab_size
    preds = np.clip(preds, 0, vocab_size - 1).astype(int)
    labels = np.clip(labels, 0, vocab_size - 1).astype(int)

    # 5. Safely decode both tokens matrices back into strings
    decoded_preds = t5_tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = t5_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 6. Format strings for SacreBLEU evaluation formatting rules
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    # 7. Compute metric weights
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [22]:
from huggingface_hub import notebook_login

notebook_login()

In [23]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_roman_urdu_to_english",
    eval_strategy="epoch",        # Evaluate at the end of each epoch
    save_strategy="epoch",              # Save a model checkpoint at the end of each epoch
    learning_rate=5e-5,                 # Standard optimal learning rate for T5 fine-tuning
    per_device_train_batch_size=8,      # If you hit a CUDA Out of Memory (OOM) error, change this to 4
    per_device_eval_batch_size=8,
    weight_decay=0.01,                  # Basic regularization to avoid overfitting
    save_total_limit=2,                 # Keeps only the 2 best checkpoints to save storage space
    num_train_epochs=3,                 # 3 epochs is ideal for this dataset size
    predict_with_generate=True,         # Enables translation text generation for validation evaluation
    fp16=True,                          # Uses Mixed Precision to speed up training on modern GPUs
    logging_steps=100,                  # Print loss values every 100 steps
    report_to="none" ,                   # Prevents third-party logging popups (like Wandb)
    push_to_hub=True
)

In [24]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=cleaned_datasets["train"],
    eval_dataset=cleaned_datasets["test"],
    data_collator=data_collator,
    processing_class=t5_tokenizer,
    compute_metrics=compute_metrics, # <-- Add this line here
)


In [25]:
# 3. Launch the fine-tuning training loop!
print("🚀 Starting fine-tuning for Roman Urdu to English...")
trainer.train()
print("🎉 Fine-tuning completed!")

🚀 Starting fine-tuning for Roman Urdu to English...


Epoch,Training Loss,Validation Loss,Bleu
1,2.016111,1.811474,11.044124
2,1.775965,1.650338,13.967133
3,1.699750,1.606815,14.542364


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1954: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Fine-tuning completed!


In [26]:
# 4. Save the finalized fine-tuned model and tokenizer locally
print("💾 Saving the trained model...")
trainer.save_model("./final_roman_urdu_to_english_model")
t5_tokenizer.save_pretrained("./final_roman_urdu_to_english_model")
print("Our model is saved and now ready for translation tasks.")

💾 Saving the trained model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Our model is saved and now ready for translation tasks.
